<a href="https://colab.research.google.com/github/jrhumberto/2026-mestrado-pep/blob/main/unir_bases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Etapa Inicial: Instalação de Pacotes

## Notebook para união de bases - Painel

>Trata-se de notebook para extração de dados porovenientes das eleições do TSE dos anos de 2018 e 2022.

**Responsável/Ano**: Humberto Bezerra de M. Júnior - 2026

## Etapa Inicial: Leitura das duas bases


#### **IMPORTANTE**: Há dois arquivos csv, um com **dimensão eleitoral** em que consta a variável "austeridade" e outro csv - **dimensão fiscal**,  com dados da gestão do mandato do eleito.

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


df_eleitoral = pd.read_csv('https://raw.githubusercontent.com/jrhumberto/2026-mestrado-pep/refs/heads/main/data/eleitoral_uf_2018_2022_austeridade.csv')

df_fiscal = pd.read_csv('https://raw.githubusercontent.com/jrhumberto/2026-mestrado-pep/refs/heads/main/data/fiscal_uf_2019_2024.csv')


# Por default, eu já crio a coluna austeridade zerada no dataset fiscal
df_fiscal['austeridade'] = 0



## Etapa 2: Geração de grupo de austeridade


>Definir quais UF eram de governadores "austeros" em mandato 2019-2022, quais UF eram com esse perfil em 2023 e 2024.

In [34]:
lista_UF_2018_austeridade_eleito = df_eleitoral[ (df_eleitoral.austeridade == 1) & (df_eleitoral.ANO_ELEICAO == 2018)]['SG_UF'].unique()
lista_UF_2022_austeridade_eleito = df_eleitoral[ (df_eleitoral.austeridade == 1) & (df_eleitoral.ANO_ELEICAO == 2022)]['SG_UF'].unique()

grupo_austeridade_eleito = {
    2018: lista_UF_2018_austeridade_eleito,
    2022: lista_UF_2022_austeridade_eleito
}


In [35]:
import numpy as np
list_year = list(df_fiscal.ano.unique())
list_uf = list(df_fiscal.uf.unique())

# Para cada UF e ano , verificar se o ano de mandato corresponde ao pleito de 2018 ou 2022 e se austero
for x in list_uf:
  for y in list_year:
    if ( ((y <= 2022) and (x in grupo_austeridade_eleito[2018]) ) or
         ((y > 2022)  and (x in grupo_austeridade_eleito[2022])) ):
      df_fiscal.loc[ (df_fiscal['uf'] == x) & (df_fiscal['ano'] == y), 'austeridade'] = 1
    else:
      continue



df_fiscal.head(50)

,uf,ano,dcl,pop,pib_prec_corr,rcl,rcla,dtp,dtp_perc_rcla,austeridade
0,AC,2019,3.116892e+09,869265,15630017000,5.357456e+09,5.357456e+09,2.878921e+09,53.74,0
1,AC,2020,3.337030e+09,881935,16476371000,5.702871e+09,5.702871e+09,3.004991e+09,52.69,0
2,AC,2021,2.847799e+09,894470,21374440000,6.690646e+09,6.651120e+09,3.421543e+09,51.44,0
3,AC,2022,2.505322e+09,906876,23676136000,7.994707e+09,7.968319e+09,3.694040e+09,46.36,0
4,AC,2023,2.051524e+09,906876,26291321000,8.573004e+09,8.496046e+09,4.113347e+09,48.41,0
5,AC,2024,1.978575e+09,829780,0,1.011123e+10,1.000246e+10,4.678325e+09,46.77,0
6,AL,2019,6.404122e+09,3322820,15630017000,8.559007e+09,8.559007e+09,3.826568e+09,44.71,0
7,AL,2020,5.813490e+09,3337357,16476371000,1.005950e+10,1.004934e+10,3.997128e+09,39.78,0
8,AL,2021,4.758264e+09,3351543,21374440000,1.252891e+10,1.246151e+10,4.436804e+09,35.60,0
9,AL,2022,7.245315e+09,3365351,23676136000,1.317791e+10,1.314893e+10,5.378105e+09,40.90,0


## Etapa Final: Criação do Painel em formato csv

In [36]:
import os

# Criar o diretório 'data' se ele não existir
os.makedirs('./data', exist_ok=True)

df_fiscal.to_csv('./data/painel_uf_2019_2024_austeridade.csv', index=False)

In [18]:
!pip install linearmodels -q

In [29]:
df_fiscal.columns

Index(['uf', 'ano', 'dcl', 'pop', 'pib_prec_corr', 'rcl', 'rcla', 'dtp',
       'dtp_perc_rcla', 'austeridade'],
      dtype='object')

In [40]:
df_fiscal = pd.read_csv('./data/painel_uf_2019_2024_austeridade.csv')

In [41]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

indice_composto = ['uf', 'ano']
df_fiscal = df_fiscal.set_index(indice_composto)

# Gestão Fiscal Responsável
df_fiscal['Y_GFR'] = (df_fiscal['dtp_perc_rcla'] < 49.0).astype(int)

# No ano 2024, não há PIB com Preços Correrntes, por isso é 0 (missing).
# Técnica de Transformar em NaN : em prol de evitar vieses.
df_fiscal['pib_prec_corr'] = df_fiscal['pib_prec_corr'].replace(0, np.nan)



In [45]:
# Função para rodar os modelos (Reaproveitamento de código)
def estimar_modelo_fixo(df, x_vars, nome_modelo):
    # Seleciona Y e X
    Y = df_fiscal['Y_GRF']
    X = df_fiscal[x_vars]

    # Adiciona constante (intercepto)
    X = sm.add_constant(X)

    # Estimação via MQO com Efeitos Fixos (within-entity)
    # Sugestão cov_type='clustered' para erros-padrão robustos agrupados por UF
    modelo = PanelOLS(Y, X, entity_effects=True, drop_absorbed=True)

    resultados = modelo.fit(cov_type='clustered', cluster_entity=True)

    variaveis = ", ".join(x_vars)
    print(f"\n{'='*30}\n{nome_modelo}\n\n Variáveis Utilizadas: {variaveis}\n{'='*30}")
    print(resultados.summary)
    return resultados

## Modelo1: Determinantes Fiscais com Nível Básico

> **Hipótese**: maiores gastos com pessoal e maior endividamento reduzem a margem do administrador e sua probabilidade de gestão responsável.


### Variáveis: dtp e dcl


In [ ]:
# Variáveis: Despesa Total de Pessoal (dtp), Dívida Consolidada Líquida (dcl)
estimar_modelo_fixo(df_fiscal, ['dtp', 'dcl'], "Modelo 1: Fiscal Básico (Níveis)")

## Modelo2:

> **Hipótese**: estados bem populosos acarretam gastos absolutos maiores, mas isso não significa  gestão irresponsável.


### Variáveis: Controle de escala nas variáveis de dtp

## Modelo3:

## Modelo 4

## Modelo5

## Modelo6:

## Modelo7:

## Modelo8:

## MOdelo9:

## Modelo10:


In [ ]:
from scipy import stats

df = endividamento.copy()

# Separar grupos
s0 = df.loc[df['austeridade'] == 0, 'dcl_valor']
s1 = df.loc[df['austeridade'] == 1, 'dcl_valor']
# Teste t com variâncias desiguais (Welch)
t_stat, p_val = stats.ttest_ind(
s1, s0,
equal_var=False,
nan_policy='omit'
)
print(f"Teste t (Welch) dcl_valor - austeridade vs outros")
print(f"t = {t_stat:.3f}, p-valor = {p_val:.4f}")
print(f"Média grupo 0 (não austeridade): {s0.mean():.2f}")
print(f"Média grupo 1 (austeridade): {s1.mean():.2f}")

Teste t (Welch) dcl_valor - austeridade vs outros
t = 5.729, p-valor = 0.0000
Média grupo 0 (não austeridade): 5332081714.27
Média grupo 1 (austeridade): 53697496310.98


In [ ]:
import statsmodels.formula.api as smf
# Criar dummies de UF e ano (efeitos fixos)
uf_d = pd.get_dummies(df['uf'], prefix='uf', drop_first=True)
ano_d = pd.get_dummies(df['ano'], prefix='ano', drop_first=True)
model_df = pd.concat([df, uf_d, ano_d], axis=1)

fe_uf = ' + '.join(uf_d.columns)
fe_ano = ' + '.join(ano_d.columns)

formula = (
'austeridade ~ dcl_valor +  pessoal_pct_rcl +' + fe_uf + ' + ' + fe_ano
)
model = smf.ols(formula=formula, data=model_df).fit(cov_type='HC3') # erros robustos
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            austeridade   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 1.357e+13
Date:                Tue, 24 Feb 2026   Prob (F-statistic):               0.00
Time:                        07:42:25   Log-Likelihood:                 2025.5
No. Observations:                 189   AIC:                            -3981.
Df Residuals:                     154   BIC:                            -3868.
Df Model:                          34                                         
Covariance Type:                  HC3                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept         6.661e-16    1.1e-05  

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 34, but rank is 33
  warnings.warn('covariance of constraints does not have full '
